In [ ]:
using Plots
using LinearAlgebra
using Revise
using May5Project
using DifferentialEquations
using OrdinaryDiffEqLowOrderRK  # for forward Euler
using OrdinaryDiffEqSDIRK       # for backward Euler

# Advection with Periodic Boundary Conditions

In [ ]:
function advection_pbc!(du, u, p, t)
    c = p[1];
    x = p[2];
    f = p[3];
    Dx = p[4];

    du .= -c * Dx * u + f.(x,t);        
    du
end

In [ ]:
x = LinRange(-5.0, 5.0, 101)[1:end-1];

u0 = exp.(-x.^2) # initial condition
tspan = (0.0, 10.0) # (t0, tmax)

c = 1;  # speed
dx = x[2] - x[1];
f = (x,t) -> 0.0; # source/sink
nx = length(x);
Dx = sparse_backwards_difference_matrix_pbc(nx, dx)

# store parameters as a tuple
p = (c, x, f, Dx);
# define and solve problem
prob_pbc = ODEProblem(advection_pbc!, u0, tspan, p)
sol_pbc = solve(prob_pbc);    

In [ ]:
maximum(diff(sol_pbc.t))

In [ ]:
t_plot = LinRange(tspan[1], tspan[2], 51);

anim = @animate for i in 1:length(t_plot)
    plot(x, sol_pbc(t_plot[i]), label="t=$(round(t_plot[i],digits=2))")
    ylims!(-1, 1)
end

gif(anim, "advection.gif", fps=15)

This attenuates.  Attentuation is mitigated by increasing the resolution.  But this takes longer to simulate.  The automatic type selection makes smaller time steps as the spatial resolution increases.

## Compare with Forward Euler, Fixed Time Step

In [ ]:
x = LinRange(-5.0, 5.0, 201)[1:end-1];

u0 = exp.(-x.^2) # initial condition
tspan = (0.0, 10.0) # (t0, tmax)

c = 1;  # speed
dx = x[2] - x[1];

dt = 0.01;
@show c * dt/dx;

f = (x,t) -> 0.0; # source/sink
nx = length(x);
Dx = sparse_backwards_difference_matrix_pbc(nx, dx)

# store parameters as a tuple
p = (c, x, f, Dx);
# define and solve problem
prob_pbc = ODEProblem(advection_pbc!, u0, tspan, p)
sol_pbc = solve(prob_pbc, Euler(); dt=dt, adaptive=false);    # explicit forward Euler with fixed time step

When `c* dt/dx <1`, there are no issues, but when it exceeds one, we see blowup.

In [ ]:
t_plot = LinRange(tspan[1], tspan[2], 51);

anim = @animate for i in 1:length(t_plot)
    plot(x, sol_pbc(t_plot[i]), label="t=$(round(t_plot[i],digits=2))")
    ylims!(-1, 1)
end

gif(anim, "advection.gif", fps=15)

# Heat Equation

## Second derivative matrix
Used for the finite difference approximation

In [ ]:
sparse_second_derivative_matrix(10, 1)

## Set up RHS and Run

In [ ]:
function heat!(du, u, p, t)
    alpha = p[1];
    dx = p[2];
    x = p[3];
    ga = p[4];
    gb = p[5];
    f = p[6];
    Dxx = p[7];
    
    du .= alpha * Dxx * u + f.(x,t);
    # apply boundary conditions
    du[1] += alpha/dx^2 * ga(t);
    du[end] += alpha/dx^2 * gb(t);
    
    du
end


In [ ]:
x = LinRange(-5.0, 5.0, 501)[2:end-1];

# u0 = exp.(-x.^2) # initial condition
# u0 = zeros(length(x)); # ice cube initial condition
# for i in 1:length(x)
#     if abs(x[i]) < 2
#         u0[i] = 1.;
#     end
# end
u0 = zeros(length(x)); # ice cube initial condition
u0[1:2:end] .= 1.0;


tspan = (0.0, 5.0) # (t0, tmax)

alpha = 1;  # diffusivity
dx = x[2] - x[1];
n = length(x);
ga = t-> 0;
gb = t-> 0;
f = (x,t) -> 0; # source/sink
Dxx = sparse_second_derivative_matrix(n, dx);

p = (alpha, dx, x, ga, gb, f, Dxx);

In [ ]:
plot(x,u0,label="t=0")

In [ ]:
# define and solve problem
prob = ODEProblem(heat!, u0, tspan, p)
sol = solve(prob);

In [ ]:
t_plot = LinRange(tspan[1], tspan[2], 1001);

anim = @animate for i in 1:length(t_plot)
    plot(x, sol(t_plot[i]), label="t=$(round(t_plot[i], digits=2))")
    ylims!(0, 1)
end
gif(anim, fps=3)

## Sources in the Heat Equation

In [ ]:
x = LinRange(-5.0, 5.0, 501)[2:end-1];

u0 = exp.(-x.^2) # initial condition
# # u0 = zeros(length(x)); # ice cube initial condition
# # for i in 1:length(x)
# #     if abs(x[i]) < 2
# #         u0[i] = 1.;
# #     end
# # end
# u0 = zeros(length(x)); # ice cube initial condition
# u0[1:2:end] .= 1.0;


tspan = (0.0, 20.0) # (t0, tmax)

alpha = 1;  # diffusivity
dx = x[2] - x[1];
n = length(x);
ga = t-> 20;
gb = t-> 2;
f = (x,t) -> 0; # source/sink
Dxx = sparse_second_derivative_matrix(n, dx);

p = (alpha, dx, x, ga, gb, f, Dxx);

In [ ]:
# define and solve problem
prob = ODEProblem(heat!, u0, tspan, p)
sol = solve(prob);

t_plot = LinRange(tspan[1], tspan[2], 101);

anim = @animate for i in 1:length(t_plot)
    plot(x, sol(t_plot[i]), label="t=$(round(t_plot[i], digits=2))")
    ylims!(0, 20)
end
gif(anim, fps=3)

In [ ]:
# define and solve problem
prob = ODEProblem(heat!, u0, tspan, p)
sol = solve(prob);


In [ ]:
t_plot = LinRange(tspan[1], tspan[2], 101);

anim = @animate for i in 1:length(t_plot)
    plot(x, sol(t_plot[i]), label="t=$(round(t_plot[i], digits=2))")
    ylims!(0, 20)
end
gif(anim, fps=60)

## Interior Source Term

In [ ]:
x = LinRange(-5.0, 5.0, 501)[2:end-1];

u0 = zeros(length(x)); # ice cube initial condition

tspan = (0.0, 50.0) # (t0, tmax)

alpha = 1;  # diffusivity
dx = x[2] - x[1];
n = length(x);
ga = t-> 0;
gb = t-> 0;
# candle near 0.
function fcandle(x,t)
    if abs(x-0.5) < 0.1
        return 5.0
    else
        return 0.0
    end
end
# f = (x,t) -> 0; # source/sink
Dxx = sparse_second_derivative_matrix(n, dx);

p = (alpha, dx, x, ga, gb, fcandle, Dxx);

In [ ]:
# define and solve problem
prob = ODEProblem(heat!, u0, tspan, p)
sol = solve(prob);


t_plot = LinRange(tspan[1], tspan[2], 101);

anim = @animate for i in 1:length(t_plot)
    plot(x, sol(t_plot[i]), label="t=$(round(t_plot[i], digits=2))")
    ylims!(0, 5)
end
gif(anim, fps=60)

# Time dependent sources
Temperature changes with time

In [ ]:
x = LinRange(-5.0, 5.0, 501)[2:end-1];

u0 = zeros(length(x)); # ice cube initial condition

tspan = (0.0, 50.0) # (t0, tmax)

alpha = 1;  # diffusivity
dx = x[2] - x[1];
n = length(x);
ga = t-> 0;
gb = t-> 0;
# candle near 0.
function fcandle(x,t)
    if abs(x) < 0.1 * cos(π*t/50)^2
        return 5.0
    else
        return 0.0
    end
end
# f = (x,t) -> 0; # source/sink
Dxx = sparse_second_derivative_matrix(n, dx);

p = (alpha, dx, x, ga, gb, fcandle, Dxx);

In [ ]:
# define and solve problem
prob = ODEProblem(heat!, u0, tspan, p)
sol = solve(prob);


t_plot = LinRange(tspan[1], tspan[2], 101);

anim = @animate for i in 1:length(t_plot)
    plot(x, sol(t_plot[i]), label="t=$(round(t_plot[i], digits=2))")
    ylims!(0, 5)
end
gif(anim, fps=60)